# Batch export of raw overnight recordings

Process every FIT file directly inside the configured sleep folder, one at a time.
Match the **actual recording timestamps** to the diary, then crop acceleration to
**[lightsoff, gotup)**. `Date` is the calendar date of lightsoff, including after
midnight. `bedtime` and `wakeup` are not required here. This is a diary-defined
time-in-bed interval, not detected sleep.

Each recording gets its own folder inside **garmin_processed_new**, with only:

- `accelerometer_samples.parquet`: `sample_time`, `accel_x`, `accel_y`, `accel_z`;
  every original sample inside the diary interval, in the original FIT scale.
- `raw_bbi.parquet`: the **full recording's original BBI snapshot fields**, one
  row per relevant FIT `record`/`session` message. Legacy slots, numbered-history
  arrays, packed payloads, counters, original invalid slots and final summaries
  remain available. Array fields become Parquet list columns. These rows are
  snapshots, not a deduplicated beat table.

Timestamps remain timezone-aware UTC in Parquet. Plots show Europe/Rome time.
Diary boundaries, recording coverage, source identity and FIT unit labels are
embedded in Parquet metadata. There is no filtering, unit conversion,
resampling, interpolation, artifact rejection, sleep scoring or HRV analysis.
The older `garmin_processed` directory is not used.


## Configuration

Use the existing `wearable` kernel (or the repository's Python 3.12 environment).
Dependencies are in `offline_processing/requirements-notebooks.txt`; this
notebook does not install packages automatically.

Inputs default to `paths.local.json` / `OVERNIGHT_FIT_PATH` and
`OVERNIGHT_DIARY_PATH`, as in the other notebooks. The parent of `fit_path`
supplies the recording folder; edit `FIT_DIR` below to change it.

With the defaults, each full-recording preview appears **before saving** and
pauses for Enter (save), `s` (skip), or `q` (stop). Set `PLOT_BEFORE_SAVE=False`
and `PAUSE_BEFORE_SAVE=False` for unattended export. `SAVE_PARQUET=False` is
preview-only. Existing exports are skipped unless `OVERWRITE=True`; use that
option to refresh exports after changing the diary.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROCESSING = next(
    (p.resolve() for p in (Path.cwd(), Path.cwd() / "offline_processing")
     if (p / "raw_batch_export.py").is_file()),
    None,
)
if PROCESSING is None:
    raise FileNotFoundError("Launch Jupyter from the repository root or offline_processing")
if str(PROCESSING) not in sys.path:
    sys.path.insert(0, str(PROCESSING))

from notebook_paths import input_path
from raw_batch_export import (
    OUTPUT_NAMES, read_raw_fit, crop_to_diary, plot_raw_preview, save_raw_tables,
)

FIT_DIR = input_path("fit_path").parent
DIARY_PATH = input_path("diary_path")
DIARY_SHEET = 0
OUTPUT_ROOT = FIT_DIR / "garmin_processed_new"
LOCAL_TZ = "Europe/Rome"

RECORDING_NAMES = None  # Or a list such as ["YYYY-MM-DD-HH-MM-SS.fit"].
SAVE_PARQUET = True
PLOT_BEFORE_SAVE = True
PAUSE_BEFORE_SAVE = True
OVERWRITE = False
STOP_ON_ERROR = False
PLOT_MAX_ACC_POINTS = 200_000  # Display only; saved acceleration is never decimated.


## Check the inputs

Only files directly inside `FIT_DIR` are processed. `bad_recs`, `giuli` and
other subfolders are excluded. The selected diary sheet must belong to the
same person. Incomplete diary rows are reported; a recording with no unique
overlapping complete interval is reported as an error rather than guessed.


In [ ]:
if not FIT_DIR.is_dir():
    raise FileNotFoundError(FIT_DIR)
if not DIARY_PATH.is_file():
    raise FileNotFoundError(DIARY_PATH)

fit_files = sorted(
    (p for p in FIT_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".fit"),
    key=lambda p: p.name.lower(),
)
if RECORDING_NAMES is not None:
    selected = set(RECORDING_NAMES)
    missing_files = selected - {p.name for p in fit_files}
    if missing_files:
        raise FileNotFoundError(f"Requested recordings not found: {sorted(missing_files)}")
    fit_files = [p for p in fit_files if p.name in selected]
if not fit_files:
    raise FileNotFoundError(f"No FIT recordings selected in {FIT_DIR}")

diary = pd.read_excel(DIARY_PATH, sheet_name=DIARY_SHEET)
diary_columns = ["Date", "lightsoff", "gotup"]
if not set(diary_columns).issubset(diary.columns):
    raise ValueError(f"Diary needs columns {diary_columns}")
incomplete = diary[diary_columns].isna().any(axis=1)
print(f"Recordings: {len(fit_files)} | complete diary rows: {(~incomplete).sum()}")
print(f"Output: {OUTPUT_ROOT}")
if incomplete.any():
    print("Incomplete diary rows (Excel row = displayed index + 2):")
    display(diary.loc[incomplete, diary_columns])
display(pd.DataFrame({"recording": [p.name for p in fit_files]}))


In [ ]:
def process_one_raw_recording(fit_path):
    output_dir = OUTPUT_ROOT / fit_path.stem
    existing = [name for name in OUTPUT_NAMES if (output_dir / name).exists()]
    if existing and not OVERWRITE:
        status = "skipped_existing" if len(existing) == len(OUTPUT_NAMES) else "skipped_partial"
        print(f"{status}: {existing}. Set OVERWRITE=True to refresh these files.")
        return {"recording": fit_path.name, "status": status}

    accel_full, raw_bbi, units = read_raw_fit(fit_path)
    accel_cropped, window = crop_to_diary(accel_full, diary, timezone=LOCAL_TZ)
    result = {
        "recording": fit_path.name,
        "diary_excel_row": window["diary_excel_row"],
        "lightsoff": window["lightsoff"], "gotup": window["gotup"],
        "full_diary_interval_recorded": window["full_diary_interval_recorded"],
        "accel_samples": len(accel_cropped), "bbi_snapshot_rows": len(raw_bbi),
    }
    print(f"Diary row {window['diary_excel_row']}: {window['lightsoff']} -> {window['gotup']}")
    print(f"ACC: {len(accel_cropped):,} / {len(accel_full):,} samples; BBI: {len(raw_bbi):,} snapshot rows")
    if not window["full_diary_interval_recorded"]:
        print("PARTIAL COVERAGE: the recording does not cover both diary boundaries.")
        print(f"Actual coverage: {window['recording_start'].tz_convert(LOCAL_TZ)} -> "
              f"{window['recording_end_exclusive'].tz_convert(LOCAL_TZ)}")
    if raw_bbi.empty:
        print("NO BBI FIELDS: the BBI export will be an empty table; acceleration is still available.")

    if PLOT_BEFORE_SAVE:
        fig = plot_raw_preview(
            accel_full, raw_bbi, window, fit_path.name,
            timezone=LOCAL_TZ, max_acc_points=PLOT_MAX_ACC_POINTS,
        )
        display(fig)
        plt.close(fig)
    if PAUSE_BEFORE_SAVE:
        action = "save" if SAVE_PARQUET else "continue (preview only)"
        while True:
            choice = input(f"Enter = {action}; s = skip; q = stop batch: ").strip().lower()
            if choice in ("", "s", "q"):
                break
        if choice == "q":
            raise KeyboardInterrupt
        if choice == "s":
            return {**result, "status": "skipped_by_user"}

    if not SAVE_PARQUET:
        return {**result, "status": "previewed"}
    paths = save_raw_tables(
        output_dir, accel_cropped, raw_bbi, window, fit_path, units,
        overwrite=OVERWRITE,
    )
    print("Saved:", ", ".join(p.name for p in paths))
    return {**result, "status": "saved" if len(raw_bbi) else "saved_no_bbi",
            "output_dir": str(output_dir)}


## Run the batch

The upper subplot shows the full raw XYZ recording; the lower subplot shows
BBI values. Dashed lines and shading mark lightsoff and gotup on both panels.
Long acceleration traces may be thinned **only for display**. Numbered BBI
snapshots are decoded only for this preview; callback times are approximate
arrival times, not physiological beat times. Legacy BBI slots use FIT record
times and may repeat between records. If decoding fails, the lower panel
explains why and the original snapshots can still be exported.

Saving retains every acceleration sample in the selected interval and every
relevant BBI snapshot in the full recording. Partial coverage is flagged;
missing data are never padded. Errors are collected in `batch_results` and
the loop continues unless `STOP_ON_ERROR=True`. Stopping during review keeps
earlier completed exports and displays the partial batch report.


In [ ]:
batch_rows = []
for number, fit_path in enumerate(fit_files, start=1):
    print(f"\n[{number}/{len(fit_files)}] {fit_path.name}")
    try:
        batch_rows.append(process_one_raw_recording(fit_path))
    except KeyboardInterrupt:
        print("Batch stopped. Earlier completed exports are retained.")
        break
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        batch_rows.append({"recording": fit_path.name, "status": "error", "error": error})
        print(error)
        if STOP_ON_ERROR:
            break

batch_results = pd.DataFrame(batch_rows)
display(batch_results)
if len(batch_results):
    display(batch_results["status"].value_counts().rename_axis("status").to_frame("recordings"))


## Read the raw exports later

```python
folder = OUTPUT_ROOT / fit_files[0].stem  # Choose a successfully exported recording.
raw_acc = pd.read_parquet(folder / "accelerometer_samples.parquet")
raw_bbi = pd.read_parquet(folder / "raw_bbi.parquet")
```

`raw_bbi` retains the original field names, including `final_` session fields.
For numbered logger schemas, reconstruct interval rows for subsequent
analyses with the existing decoder:

```python
from raw_batch_export import snapshot_messages
from fit_bbi import decode_snapshots
interval_rows, decode_report = decode_snapshots(snapshot_messages(raw_bbi))
intervals = pd.DataFrame(interval_rows)
```

Read `decode_report` before analyzing recovered intervals: inconsistent
snapshots, sequence gaps and invalid slots remain explicit. Legacy slots
cannot establish unique beat sequence numbers or exact beat times.

To inspect the provenance and diary interval without loading the samples:

```python
import json
import pyarrow.parquet as pq
metadata = json.loads(pq.read_schema(folder / "accelerometer_samples.parquet")
                      .metadata[b"overnight_raw_export"])
metadata["window"]
```
